# Pipeline Check -- Phase 2 only (fast-iteration companion)

Same idea as `kaggle_check.ipynb`, minus the feasibility check, Phase 1 baselines, and Phase 1b ablation -- those don't gate Phase 2 at all (no Phase 2 stage's `required_paths` points at anything they produce) and, on a prior run, either already passed or hit an already-understood expected failure (Gemma 4 E4B OOMing at full precision, which Phase 1b's own markdown already calls out as expected). Re-running them just to get back to whatever's still actually being debugged in Phase 2 cost ~40-50 minutes per iteration for zero new information -- this notebook skips straight to Phase 2, at the same tiny check-scale (2-16 samples/tasks, 2-3 GRPO steps) as `kaggle_check.ipynb`.

Once Phase 2 is passing clean here, go back to `kaggle_check.ipynb` for one full run top-to-bottom before trusting `kaggle_train.ipynb` -- this notebook is for fast bug iteration, not a substitute for the full check.

Same before-running requirements (GPU T4 x2, Internet On, `GITHUB_TOKEN` secret) and the same `git submodule update --init` for `alphamaze_reference` -- still needed since Phase 2's own eval cells score post-training checkpoints on MazeBench too, not just GridRoute.

Generated from the same `build_notebook.py` as the other two notebooks (`build_cells(check=True, phase2_only=True)`), so it can't structurally drift from them -- if Phase 2's stages change, regenerate all three.

In [6]:
import os, subprocess, sys

REPO_URL = "github.com/Vedang-P/neuro-symbolic-pathfinding.git"
REPO_DIR = "/kaggle/working/neuro-symbolic-pathfinding"

# GIT_TERMINAL_PROMPT=0: if auth still fails for some other reason (token
# lacks access to this repo, expired, etc.), git fails immediately with a
# clear error instead of hanging on an interactive password prompt that has
# nowhere to go in a notebook environment.
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True, env=env)
elif not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        # Token must go in the PASSWORD slot (after the colon), not as a bare
        # username before @ -- a bare-username URL leaves git still needing a
        # password, which it then can't prompt for here ("could not read
        # Password ... No such device or address"). "x-access-token" as the
        # username is GitHub's documented convention for token-based HTTPS auth.
        clone_url = f"https://x-access-token:{token}@{REPO_URL}"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True, env=env)
        print("Cloned via GITHUB_TOKEN secret.")
    except Exception as e:
        print(f"Could not clone via secret ({type(e).__name__}: {e}).")
        print("Check: the secret is attached to this notebook (Add-ons -> Secrets -> make sure")
        print("it's toggled ON for this notebook specifically), the token hasn't expired, and it")
        print("has read access to this exact repo (fine-grained tokens need per-repo access grants).")
        print("If you attached the repo as a Kaggle Dataset instead, set REPO_DIR above to its mount")
        print("path (typically /kaggle/input/<dataset-name>) and re-run this cell.")
        raise

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())


Repo already present, pulling latest...
Already up to date.
Working dir: /kaggle/working/neuro-symbolic-pathfinding


In [7]:
# Pulls in alphamaze_reference/ (github.com/menloresearch/visual-thinker) -- eval.py uses their
# real MazeBench scoring code directly from this submodule when present, falling back to a less
# faithful exact-match approach (with a loud warning) if it isn't.
subprocess.run(["git", "submodule", "update", "--init"], check=True)


CompletedProcess(args=['git', 'submodule', 'update', '--init'], returncode=0)

In [8]:
# Optional: read from Kaggle Secrets, never pasted here -- higher HF Hub rate
# limits/download speed, and required if any candidate model is gated.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some older library versions check this name
    print("HF_TOKEN set from Kaggle secret.")
except Exception as e:
    print(f"No HF_TOKEN secret found ({type(e).__name__}) -- proceeding unauthenticated "
          "(fine for these public models, just slower/rate-limited).")


HF_TOKEN set from Kaggle secret.


In [9]:
# requirements.txt pins peft>=0.19.0, which ships its own Gemma4ClippableLinear
# support (see hf_models.py's load_trainable_model() docstring) -- no
# separate unsloth install needed here.
%pip install -q -r requirements.txt
# Kaggle's preinstalled wandb has been observed with its SDK and its own
# bundled generated protobuf file out of sync ("cannot import name 'Imports'
# from wandb.proto.wandb_telemetry_pb2"), which crashes on import -- trl's
# base_trainer.py checks wandb availability at import time, unconditionally.
# We never use wandb (report_to=[] everywhere) -- removing it outright is
# the surest fix, on top of WANDB_DISABLED below.
%pip uninstall -q -y wandb


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
# Belt and suspenders alongside hf_models.configure_quiet_logging() (which
# also sets these for every subprocess this notebook launches) -- set here
# too in case anything in the notebook's own kernel process ever imports
# wandb directly.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Run infrastructure: `run_stage()`

Every pipeline stage below goes through this. It never raises -- a stage that OOMs, hits a bug, or has a transient HF Hub hiccup is logged and skipped over, not fatal to the rest of an unattended run. `required_paths` lets a stage declare "don't even try me if this input is missing" (e.g. a GRPO stage needing the preceding SFT stage's checkpoint), so a failure shows up as a clear, attributable skip reason in the log instead of a cryptic error three layers deep in someone else's library -- every `required_paths` entry below points at a specific file inside a checkpoint dir (`tokenizer_config.json`, the last thing `train_sft.py`/`train_grpo.py` write), not the bare directory: `TrainingArguments` creates `output_dir` immediately on construction, long before training actually runs, so a stage that OOMs mid-training still leaves that directory sitting on disk -- checking for the directory alone would let a downstream stage "pass" the check and then fail somewhere far more confusing (a checkpoint load with no actual tokenizer files in it). `steps_for_budget()` reads a prior GRPO timing-test's measured per-step cost and sizes the following full run to fit a wall-clock budget automatically, which is what makes "no human between cells" actually work -- without it, the standard advice ("check the timing printout, adjust --max_steps") requires exactly the babysitting this notebook is built to avoid.

In [11]:
import json
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("./results_check_phase2")
STAGE_LOG_PATH = RESULTS_DIR / "stage_log.json"
stage_results = json.loads(STAGE_LOG_PATH.read_text()) if STAGE_LOG_PATH.exists() else []

# Safety-ceiling timeouts (a hard kill-switch if something hangs) -- these are
# NOT the target duration for a stage, just an upper bound. Full GRPO runs
# size their own --max_steps well below their timeout via steps_for_budget().
TIMEOUT_QUICK = 10 * 60        # feasibility check, GRPO timing tests
# This check notebook uses a tiny --n (2) on every benchmark call, so
# even MazeBench's worst-case ~180s/maze (measured on real hardware at 8192
# max_new_tokens, see kaggle_train.ipynb) keeps every eval call well under
# TIMEOUT_EVAL. Not trying to reproduce a meaningful accuracy number here --
# only to prove every stage runs end to end without error.
TIMEOUT_EVAL = 15 * 60         # baseline/checkpoint evals
TIMEOUT_SFT = 20 * 60          # SFT warm-start
TIMEOUT_GRPO_FULL = 20 * 60  # full GRPO training runs, ceiling above their own time budget

# Target wall-clock budget per full GRPO training run -- lower this if your
# weekly quota is tight, raise it if you have room. Applied per condition, so
# the pricier "consistency" condition naturally gets fewer steps than
# "single"/"mixed" for the same budget, without any special-casing.
GRPO_BUDGET_MINUTES = 2


def _save_stage_log():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    STAGE_LOG_PATH.write_text(json.dumps(stage_results, indent=2))


def run_stage(name, args, required_paths=None, timeout=None):
    """Run `python <args>` as a subprocess. Always returns, never raises."""
    print(f"\n{'='*70}\n\u25b6 {name}\n{'='*70}")
    for p in (required_paths or []):
        if not Path(p).exists():
            print(f"\u23ed  SKIPPED -- required input not found: {p}")
            stage_results.append({"stage": name, "status": "skipped", "reason": f"missing {p}"})
            _save_stage_log()
            return stage_results[-1]

    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable] + args, timeout=timeout)
        elapsed_min = round((time.time() - t0) / 60, 1)
        if proc.returncode == 0:
            print(f"\n\u2705 DONE ({elapsed_min} min)")
            result = {"stage": name, "status": "ok", "elapsed_min": elapsed_min}
        else:
            print(f"\n\u274c FAILED (exit code {proc.returncode}, {elapsed_min} min) -- see output above")
            result = {"stage": name, "status": "failed", "exit_code": proc.returncode, "elapsed_min": elapsed_min}
    except subprocess.TimeoutExpired:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u23f1  TIMED OUT after {elapsed_min} min (limit {timeout/60:.0f} min)")
        result = {"stage": name, "status": "timeout", "elapsed_min": elapsed_min}
    except Exception as e:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u274c FAILED (exception: {e})")
        result = {"stage": name, "status": "error", "error": str(e), "elapsed_min": elapsed_min}

    stage_results.append(result)
    _save_stage_log()
    return result


def steps_for_budget(timing_dir, budget_minutes=GRPO_BUDGET_MINUTES, min_steps=3, default_steps=3):
    """Read a prior train_grpo.py run's timing.json and compute a step count
    that fits budget_minutes at that measured per-step cost. Falls back to
    default_steps if timing.json is missing (e.g. the timing test itself
    failed) -- better to attempt a bounded full run than skip it outright."""
    timing_path = Path(timing_dir) / "timing.json"
    if not timing_path.exists():
        print(f"  (no timing.json at {timing_dir}, using default {default_steps} steps)")
        return default_steps
    t = json.loads(timing_path.read_text())
    per_step = t.get("per_step_s", 0)
    if per_step <= 0:
        return default_steps
    steps = max(min_steps, int(budget_minutes * 60 / per_step))
    print(f"  (timing.json: {per_step:.1f}s/step -> {steps} steps fits a {budget_minutes}min budget)")
    return steps


## Phase 1 / Phase 1b -- skipped

See this notebook's intro: already covered by a prior `kaggle_check.ipynb` (or `kaggle_train.ipynb`) run. `ALPHAMAZE_LOCAL_PATH` is intentionally not defined here since nothing below needs it -- Phase 2 never evaluates the `alphamaze` model, only `gemma4-e2b` checkpoints (including on MazeBench, which is why `alphamaze_reference` is still cloned above).

## Phase 2, recipe 1: single-format (GridRoute NL only) on Gemma 4 E2B

SFT warm-start, a short GRPO timing test, then a full run sized to fit `GRPO_BUDGET_MINUTES` at the timing test's measured per-step cost -- see the run-infrastructure cell above for why this is computed rather than manually tuned.

In [12]:
SFT_SINGLE_DIR = "./results_check_phase2/sft_gemma4-e2b_single"
run_stage("SFT: single-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "nl", "--grid_size", "5",
           "--n_tasks", "16", "--epochs", "1", "--output_dir", SFT_SINGLE_DIR], timeout=TIMEOUT_SFT)



▶ SFT: single-format (Gemma 4 E2B)
Model: google/gemma-4-E2B-it
Format: nl  Tasks: 16  Grid: 5x5  Epochs: 1  LR: 1e-05


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 504.97it/s] 


Model + LoRA ready (backend=bnb_peft)
Building SFT dataset (nl, 16 tasks, 5x5)...
  16 training rows.


Truncating train dataset: 100%|██████████| 16/16 [00:00<00:00, 2416.16 examples/s]


Starting SFT: 1 epoch(s), 16 examples
{'train_runtime': '16.44', 'train_samples_per_second': '0.973', 'train_steps_per_second': '0.122', 'train_loss': '3.494', 'entropy': '0.1474', 'num_tokens': '2160', 'mean_token_accuracy': '0.6494', 'epoch': '1'}
SFT complete in 0.3 min (0.0h)
Model saved to ./results_check_phase2/sft_gemma4-e2b_single

✅ DONE (1.6 min)


{'stage': 'SFT: single-format (Gemma 4 E2B)',
 'status': 'ok',
 'elapsed_min': 1.6}

In [13]:
GRPO_SINGLE_TIMING_DIR = "./results_check_phase2/grpo_gemma4-e2b_single_timing"
run_stage("GRPO timing test: single-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "256",
           "--output_dir", GRPO_SINGLE_TIMING_DIR],
          required_paths=[SFT_SINGLE_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: single-format
⚠️  NOTE: 'gemma4-e2b' needs ~9+GB and won't fit a 6GB laptop GPU. (fine on Kaggle's 16GB T4)
Loading google/gemma-4-E2B-it (4bit=True, seq_len=2048)... + adapter ./results_check_phase2/sft_gemma4-e2b_single


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 565.22it/s] 


  [load_trainable_model] loading saved adapter from ./results_check_phase2/sft_gemma4-e2b_single (is_trainable=True) on top of base google/gemma-4-E2B-it
Model + LoRA ready in 8.6s (backend=bnb_peft)
Building 'single' dataset (8 GridRoute 5x5 problems)...
  8 training rows.

Starting GRPO: condition=single, 2 steps, group size 4, 8 training rows.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-06', 'num_tokens': '1021', 'completions/mean_length': '29.62', 'completions/min_length': '24', 'completions/max_length': '63', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '29.62', 'completions/min_terminated_length': '24', 'completions/max_terminated_length': '63', 'rewards/reward_fn/mean': '-0.5', 'rewards/reward_fn/std': '0', 'reward': '-0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.1285', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time

{'stage': 'GRPO timing test: single-format',
 'status': 'ok',
 'elapsed_min': 2.0}

In [14]:
GRPO_SINGLE_DIR = "./results_check_phase2/grpo_gemma4-e2b_single"
single_steps = steps_for_budget(GRPO_SINGLE_TIMING_DIR)
run_stage(f"GRPO: single-format full run ({single_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(single_steps), "--max_completion_length", "256", "--output_dir", GRPO_SINGLE_DIR],
          required_paths=[SFT_SINGLE_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_GRPO_FULL)


  (timing.json: 48.9s/step -> 3 steps fits a 2min budget)

▶ GRPO: single-format full run (3 steps)
⚠️  NOTE: 'gemma4-e2b' needs ~9+GB and won't fit a 6GB laptop GPU. (fine on Kaggle's 16GB T4)
Loading google/gemma-4-E2B-it (4bit=True, seq_len=2048)... + adapter ./results_check_phase2/sft_gemma4-e2b_single


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 562.49it/s]


  [load_trainable_model] loading saved adapter from ./results_check_phase2/sft_gemma4-e2b_single (is_trainable=True) on top of base google/gemma-4-E2B-it
Model + LoRA ready in 8.1s (backend=bnb_peft)
Building 'single' dataset (8 GridRoute 5x5 problems)...
  8 training rows.

Starting GRPO: condition=single, 3 steps, group size 4, 8 training rows.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-06', 'num_tokens': '1021', 'completions/mean_length': '29.62', 'completions/min_length': '24', 'completions/max_length': '63', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '29.62', 'completions/min_terminated_length': '24', 'completions/max_terminated_length': '63', 'rewards/reward_fn/mean': '-0.5', 'rewards/reward_fn/std': '0', 'reward': '-0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.1285', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time

{'stage': 'GRPO: single-format full run (3 steps)',
 'status': 'ok',
 'elapsed_min': 3.4}

In [15]:
run_stage("Eval single-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "mazebench", "--n", "2", "--max_new_tokens", "1024",
           "--output_dir", "./results_check_phase2/eval"],
          required_paths=[GRPO_SINGLE_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_EVAL)
run_stage("Eval single-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check_phase2/eval"],
          required_paths=[GRPO_SINGLE_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_EVAL)



▶ Eval single-format checkpoint: MazeBench
Model: google/gemma-4-E2B-it  + adapter: ./results_check_phase2/grpo_gemma4-e2b_single
4bit=True  backend=hf  benchmark=mazebench


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 557.35it/s]


Results saved incrementally to: results_check_phase2/eval/gemma4-e2b_mazebench_20260716_171655.json


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
thought
The user wants me to solve a maze based on a specific set of rules.
I need to track the current position, the origin, the target, and the maze structure (walls).
The maze is represented by tokens: `<|row-col|><|wall_token|>`

**Maze Parsing:**

Row 0:
<|0-0|><|up_down_left_wall|><|blank|> (0,0)
<|0-1|><|up_down_wall|><|blank|> (0,1)
<|0-2|><|up_down_wall|><|blank|> (0,2)
<|0-3|><|up_down_wall|><|blank|> (0,3)
<|0-4|><|up_right_wall|><|blank|> (0,4)

Row 1:
<|1-0|><|up_left_wall|><|blank|> (1,0)
<|1-1|><|up_down_wall|><|blank|> (1,1)
<|1-2|><|up_down_wall|><|origin|> (1,2) -> Origin is at (1,2)
<|1-3|><|up_right_wall|><|blank|> (1,3)
<|1-4|><|left_right_wall|><|blank|> (1,4)

Row 2:
<|2-0|><|down_left_wall|><|blank|> (2,0)
<|2-1|><|up_right_wall|><|blank|> (2,1)
<|2-2|><|up_left_wall|><|blank|> (2,2)
<|2-3|><|down_right_wall|><|blank|> (2,3)
<|2-4|><|left_right_wall|><|blank|> (2,4)

Row 3:
<|3-0|><|up_left_right_wall|><|blank|> (3,0)
<|3-1

...

GridRoute-nl-5x5:  50%|█████     | 1/2 [03:23<03:23, 203.60s/task, optimal=1/1, valid=1/1]


  [1/2] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2,0) to the target point (4,3) on a 5x5 grid.
The grid coordinates are (row, col), where rows are 0 to 4 and columns are 0 to 4.
The obstacles are at [(0,1)].
Allowed moves are up, down, left, right.

Start: S = (2, 0)
Target: T = (4, 3)
Grid size: 5x5.

Obstacles: O = {(0, 1)}

I need to find a sequence of valid moves. Since the problem doesn't specify if I need the shortest path, I will aim for a simple, valid path.

Let's represent the grid and track visited cells to avoid loops, although for a simple path finding, just ensuring validity is key.

Grid visualization (R, C):
(0,0) (0,1)X (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Start (2, 0). Target (4, 3).

Path planning:
1. From (2, 0). Possible moves: (1, 0), (3, 0), (2, 1).
2. Let's 

GridRoute-nl-5x5: 100%|██████████| 2/2 [05:44<00:00, 172.15s/task, optimal=2/2, valid=2/2]



✅ DONE (6.1 min)


{'stage': 'Eval single-format checkpoint: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 6.1}

## Phase 2, recipe 2: mixed-format (NL + token, naive)

Does training on both formats (interleaved, same underlying grids) do better than single-format alone -- for either benchmark?

In [16]:
SFT_MIXED_DIR = "./results_check_phase2/sft_gemma4-e2b_mixed"
run_stage("SFT: mixed-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "mixed", "--grid_size", "5",
           "--n_tasks", "16", "--epochs", "1", "--output_dir", SFT_MIXED_DIR], timeout=TIMEOUT_SFT)



▶ SFT: mixed-format (Gemma 4 E2B)
Model: google/gemma-4-E2B-it
Format: mixed  Tasks: 16  Grid: 5x5  Epochs: 1  LR: 1e-05


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 541.53it/s] 


Model + LoRA ready (backend=bnb_peft)
Building SFT dataset (mixed, 16 tasks, 5x5)...
  32 training rows.


Truncating train dataset: 100%|██████████| 32/32 [00:00<00:00, 2342.33 examples/s]


Starting SFT: 1 epoch(s), 32 examples
{'train_runtime': '78.25', 'train_samples_per_second': '0.409', 'train_steps_per_second': '0.051', 'train_loss': '4.869', 'entropy': '0.2005', 'num_tokens': '1.026e+04', 'mean_token_accuracy': '0.5592', 'epoch': '1'}
SFT complete in 1.3 min (0.0h)
Model saved to ./results_check_phase2/sft_gemma4-e2b_mixed

✅ DONE (1.8 min)


{'stage': 'SFT: mixed-format (Gemma 4 E2B)',
 'status': 'ok',
 'elapsed_min': 1.8}

In [17]:
GRPO_MIXED_TIMING_DIR = "./results_check_phase2/grpo_gemma4-e2b_mixed_timing"
run_stage("GRPO timing test: mixed-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "256",
           "--output_dir", GRPO_MIXED_TIMING_DIR],
          required_paths=[SFT_MIXED_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: mixed-format
⚠️  NOTE: 'gemma4-e2b' needs ~9+GB and won't fit a 6GB laptop GPU. (fine on Kaggle's 16GB T4)
Loading google/gemma-4-E2B-it (4bit=True, seq_len=2048)... + adapter ./results_check_phase2/sft_gemma4-e2b_mixed


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 549.24it/s]


  [load_trainable_model] loading saved adapter from ./results_check_phase2/sft_gemma4-e2b_mixed (is_trainable=True) on top of base google/gemma-4-E2B-it
Model + LoRA ready in 8.3s (backend=bnb_peft)
Building 'mixed' dataset (8 GridRoute 5x5 problems)...
  16 training rows.

Starting GRPO: condition=mixed, 2 steps, group size 4, 16 training rows.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-06', 'num_tokens': '1237', 'completions/mean_length': '56.62', 'completions/min_length': '24', 'completions/max_length': '256', 'completions/clipped_ratio': '0.125', 'completions/mean_terminated_length': '28.14', 'completions/min_terminated_length': '24', 'completions/max_terminated_length': '48', 'rewards/reward_fn/mean': '-0.5', 'rewards/reward_fn/std': '0', 'reward': '-0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.079', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_t

{'stage': 'GRPO timing test: mixed-format', 'status': 'ok', 'elapsed_min': 3.1}

In [18]:
GRPO_MIXED_DIR = "./results_check_phase2/grpo_gemma4-e2b_mixed"
mixed_steps = steps_for_budget(GRPO_MIXED_TIMING_DIR)
run_stage(f"GRPO: mixed-format full run ({mixed_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(mixed_steps), "--max_completion_length", "256", "--output_dir", GRPO_MIXED_DIR],
          required_paths=[SFT_MIXED_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_GRPO_FULL)


  (timing.json: 79.1s/step -> 3 steps fits a 2min budget)

▶ GRPO: mixed-format full run (3 steps)
⚠️  NOTE: 'gemma4-e2b' needs ~9+GB and won't fit a 6GB laptop GPU. (fine on Kaggle's 16GB T4)
Loading google/gemma-4-E2B-it (4bit=True, seq_len=2048)... + adapter ./results_check_phase2/sft_gemma4-e2b_mixed


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 531.69it/s]


  [load_trainable_model] loading saved adapter from ./results_check_phase2/sft_gemma4-e2b_mixed (is_trainable=True) on top of base google/gemma-4-E2B-it
Model + LoRA ready in 8.7s (backend=bnb_peft)
Building 'mixed' dataset (8 GridRoute 5x5 problems)...
  16 training rows.

Starting GRPO: condition=mixed, 3 steps, group size 4, 16 training rows.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-06', 'num_tokens': '1237', 'completions/mean_length': '56.62', 'completions/min_length': '24', 'completions/max_length': '256', 'completions/clipped_ratio': '0.125', 'completions/mean_terminated_length': '28.14', 'completions/min_terminated_length': '24', 'completions/max_terminated_length': '48', 'rewards/reward_fn/mean': '-0.5', 'rewards/reward_fn/std': '0', 'reward': '-0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.079', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_t

{'stage': 'GRPO: mixed-format full run (3 steps)',
 'status': 'ok',
 'elapsed_min': 4.5}

In [19]:
run_stage("Eval mixed-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "mazebench", "--n", "2", "--max_new_tokens", "1024",
           "--output_dir", "./results_check_phase2/eval"],
          required_paths=[GRPO_MIXED_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_EVAL)
run_stage("Eval mixed-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check_phase2/eval"],
          required_paths=[GRPO_MIXED_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_EVAL)



▶ Eval mixed-format checkpoint: MazeBench
Model: google/gemma-4-E2B-it  + adapter: ./results_check_phase2/grpo_gemma4-e2b_mixed
4bit=True  backend=hf  benchmark=mazebench


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 525.98it/s] 


Results saved incrementally to: results_check_phase2/eval/gemma4-e2b_mazebench_20260716_174609.json


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
thought
The user wants me to solve a maze based on a specific set of rules.
I need to track the current position, the origin, the target, and the maze structure (walls).

Maze representation:
Coordinates are <|row-col|>.
Tokens are: Coordinate, Wall description, Movement, Origin, Target.

Let's parse the maze structure first.

Grid size: 5 rows (0-4), 4 columns (0-3).

Maze input:
R0: <|0-0|><|up_down_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_down_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
R1: <|1-0|><|up_left_wall|><|blank|><|1-1|><|up_down_wall|><|blank|><|1-2|><|up_down_wall|><|origin|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
R2: <|2-0|><|down_left_wall|><|blank|><|2-1|><|up_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><|2-3|><|down_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
R3: <|3-0|><|up_left_right_wall|><|blank|><|3-1|><|left_right_

...

GridRoute-nl-5x5:  50%|█████     | 1/2 [02:55<02:55, 175.03s/task, optimal=1/1, valid=1/1]


  [1/2] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2,0) to the target point (4,3) on a 5x5 grid.
The grid coordinates are (row, col), where rows are 0 to 4 and columns are 0 to 4.
The obstacles are at [(0,1)].
Allowed moves are up, down, left, right.

Start: S = (2, 0)
Target: T = (4, 3)
Grid size: 5x5.

Obstacles: O = {(0, 1)}

I will use Breadth-First Search (BFS) or simply try to find a path, keeping track of visited cells.

Grid representation (R, C):
(0,0) (0,1)X (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Start (2, 0). Target (4, 3).

Step 1: Start at (2, 0).
Possible moves:
1. (1, 0)
2. (3, 0)
3. (2, 1)

Let's try to move towards the target (4, 3). We need to increase row index and column index.

Path attempt 1:
1. (2, 0) -> (3, 0) (Moving down)
2. (3, 0) -> (4, 0) (Moving down)
3. (

GridRoute-nl-5x5: 100%|██████████| 2/2 [05:25<00:00, 162.99s/task, optimal=2/2, valid=2/2]



✅ DONE (5.8 min)


{'stage': 'Eval mixed-format checkpoint: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 5.8}

## Phase 2, recipe 3: consistency-reward (one candidate recipe, not this project's headline claim)

Adapts Elhady et al.'s cross-lingual consistency-reward mechanism to cross-format spatial reasoning -- try it, report honestly whether it beats `mixed` or not. See `train_grpo.py`'s `make_reward_fn` docstring for the exact mechanism. This condition generates an extra partner completion per reward call (roughly double the per-step cost of single/mixed) -- `steps_for_budget()` accounts for that automatically since it reads THIS condition's own timing test, not single/mixed's.

In [20]:
SFT_CONSISTENCY_DIR = SFT_MIXED_DIR  # consistency reuses the same mixed-format SFT warm-start
GRPO_CONSISTENCY_TIMING_DIR = "./results_check_phase2/grpo_gemma4-e2b_consistency_timing"
run_stage("GRPO timing test: consistency-reward",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "256",
           "--output_dir", GRPO_CONSISTENCY_TIMING_DIR],
          required_paths=[SFT_CONSISTENCY_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: consistency-reward
⚠️  NOTE: 'gemma4-e2b' needs ~9+GB and won't fit a 6GB laptop GPU. (fine on Kaggle's 16GB T4)
Loading google/gemma-4-E2B-it (4bit=True, seq_len=2048)... + adapter ./results_check_phase2/sft_gemma4-e2b_mixed


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 549.74it/s]


  [load_trainable_model] loading saved adapter from ./results_check_phase2/sft_gemma4-e2b_mixed (is_trainable=True) on top of base google/gemma-4-E2B-it
Model + LoRA ready in 8.4s (backend=bnb_peft)
Building 'consistency' dataset (8 GridRoute 5x5 problems)...
  16 training rows.

Starting GRPO: condition=consistency, 2 steps, group size 4, 16 training rows.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-06', 'num_tokens': '1237', 'completions/mean_length': '56.62', 'completions/min_length': '24', 'completions/max_length': '256', 'completions/clipped_ratio': '0.125', 'completions/mean_terminated_length': '28.14', 'completions/min_terminated_length': '24', 'completions/max_terminated_length': '48', 'rewards/reward_fn/mean': '-0.5', 'rewards/reward_fn/std': '0', 'reward': '-0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.079', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': 

{'stage': 'GRPO timing test: consistency-reward',
 'status': 'ok',
 'elapsed_min': 3.5}

In [21]:
GRPO_CONSISTENCY_DIR = "./results_check_phase2/grpo_gemma4-e2b_consistency"
consistency_steps = steps_for_budget(GRPO_CONSISTENCY_TIMING_DIR)
run_stage(f"GRPO: consistency-reward full run ({consistency_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(consistency_steps), "--max_completion_length", "256", "--output_dir", GRPO_CONSISTENCY_DIR],
          required_paths=[SFT_CONSISTENCY_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_GRPO_FULL)


  (timing.json: 93.1s/step -> 3 steps fits a 2min budget)

▶ GRPO: consistency-reward full run (3 steps)
⚠️  NOTE: 'gemma4-e2b' needs ~9+GB and won't fit a 6GB laptop GPU. (fine on Kaggle's 16GB T4)
Loading google/gemma-4-E2B-it (4bit=True, seq_len=2048)... + adapter ./results_check_phase2/sft_gemma4-e2b_mixed


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 540.13it/s]


  [load_trainable_model] loading saved adapter from ./results_check_phase2/sft_gemma4-e2b_mixed (is_trainable=True) on top of base google/gemma-4-E2B-it
Model + LoRA ready in 8.6s (backend=bnb_peft)
Building 'consistency' dataset (8 GridRoute 5x5 problems)...
  16 training rows.

Starting GRPO: condition=consistency, 3 steps, group size 4, 16 training rows.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-06', 'num_tokens': '1237', 'completions/mean_length': '56.62', 'completions/min_length': '24', 'completions/max_length': '256', 'completions/clipped_ratio': '0.125', 'completions/mean_terminated_length': '28.14', 'completions/min_terminated_length': '24', 'completions/max_terminated_length': '48', 'rewards/reward_fn/mean': '-0.5', 'rewards/reward_fn/std': '0', 'reward': '-0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.079', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': 

{'stage': 'GRPO: consistency-reward full run (3 steps)',
 'status': 'ok',
 'elapsed_min': 9.8}

In [22]:
run_stage("Eval consistency-reward checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "mazebench", "--n", "2", "--max_new_tokens", "1024",
           "--output_dir", "./results_check_phase2/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_EVAL)
run_stage("Eval consistency-reward checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check_phase2/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR + "/tokenizer_config.json"], timeout=TIMEOUT_EVAL)



▶ Eval consistency-reward checkpoint: MazeBench
Model: google/gemma-4-E2B-it  + adapter: ./results_check_phase2/grpo_gemma4-e2b_consistency
4bit=True  backend=hf  benchmark=mazebench


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 547.38it/s]


Results saved incrementally to: results_check_phase2/eval/gemma4-e2b_mazebench_20260716_182020.json


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
thought
The user wants me to solve a maze based on a specific set of rules.
I need to track the current position, the origin, the target, and the maze structure (walls).

Maze representation:
Coordinates are <|row-col|>.
Tokens are: Coordinate, Wall description, Movement, Origin, Target.

Let's parse the maze structure first.

Grid size: 5 rows (0-4), 4 columns (0-3).

Maze input:
R0: <|0-0|><|up_down_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_down_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
R1: <|1-0|><|up_left_wall|><|blank|><|1-1|><|up_down_wall|><|blank|><|1-2|><|up_down_wall|><|origin|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
R2: <|2-0|><|down_left_wall|><|blank|><|2-1|><|up_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><|2-3|><|down_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
R3: <|3-0|><|up_left_right_wall|><|blank|><|3-1|><|left_right_

...

GridRoute-nl-5x5:  50%|█████     | 1/2 [02:54<02:54, 174.43s/task, optimal=1/1, valid=1/1]


  [1/2] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2,0) to the target point (4,3) on a 5x5 grid.
The grid coordinates are (row, col), where rows are 0 to 4 and columns are 0 to 4.
The obstacles are at [(0,1)].
Allowed moves are up, down, left, right.

Start: S = (2, 0)
Target: T = (4, 3)
Grid size: 5x5.

Obstacles: O = {(0, 1)}

I will use Breadth-First Search (BFS) or simply try to find a path, keeping track of visited cells.

Grid representation (R, C):
(0,0) (0,1)X (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Start (2, 0). Target (4, 3).

Step 1: Start at (2, 0).
Possible moves:
1. (1, 0)
2. (3, 0)
3. (2, 1)

Let's try to move towards the target (4, 3). We need to increase row index and column index.

Path attempt 1:
1. (2, 0) -> (3, 0) (Moving down)
2. (3, 0) -> (4, 0) (Moving down)
3. (

GridRoute-nl-5x5: 100%|██████████| 2/2 [05:23<00:00, 161.71s/task, optimal=2/2, valid=2/2]



✅ DONE (5.8 min)


{'stage': 'Eval consistency-reward checkpoint: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 5.8}

## Run summary

At-a-glance status of every stage attempted -- read this first when checking back on an unattended run, before digging into the full cell outputs above.

In [23]:
import pandas as pd

summary_df = pd.DataFrame(stage_results)
if not summary_df.empty:
    ok = (summary_df["status"] == "ok").sum()
    print(f"{ok}/{len(summary_df)} stages completed successfully.\n")
pd.set_option("display.max_colwidth", None)
summary_df


12/14 stages completed successfully.



,stage,status,elapsed_min
0,SFT: single-format (Gemma 4 E2B),ok,1.6
1,GRPO timing test: single-format,ok,2.0
2,GRPO: single-format full run (3 steps),ok,3.4
3,Eval single-format checkpoint: MazeBench,ok,13.8
4,Eval single-format checkpoint: GridRoute NL,ok,6.1
5,SFT: mixed-format (Gemma 4 E2B),ok,1.8
6,GRPO timing test: mixed-format,ok,3.1
7,GRPO: mixed-format full run (3 steps),ok,4.5
8,Eval mixed-format checkpoint: MazeBench,timeout,15.0
9,Eval mixed-format checkpoint: GridRoute NL,ok,5.8


## Pipeline check verdict

`run_stage()` deliberately never raises -- that's what lets an unattended multi-hour real run survive one bad stage. But this notebook exists to answer one question (did everything work?), so this cell inverts that: it fails loudly, on purpose, the moment anything didn't complete cleanly.

In [24]:
failed = [r for r in stage_results if r["status"] != "ok"]

print(f"\n{'='*70}")
if failed:
    print(f"CHECK FAILED -- {len(failed)}/{len(stage_results)} stage(s) did not complete cleanly:\n")
    for r in failed:
        detail = r.get("error") or (f"exit code {r['exit_code']}" if "exit_code" in r else r.get("reason", ""))
        print(f"  [{r['status'].upper()}] {r['stage']}" + (f" -- {detail}" if detail else ""))
    print(f"{'='*70}")
    raise RuntimeError(
        f"{len(failed)} pipeline stage(s) failed or were skipped -- see the list above and the "
        "full cell output higher in this notebook for the actual error/traceback. Fix before "
        "trusting a real kaggle_train.ipynb run."
    )
else:
    print(f"CHECK PASSED -- all {len(stage_results)} attempted stages completed successfully.")
    print("Covered: Phase 2 recipes 1-3, eval (feasibility/Phase 1/Phase 1b intentionally skipped, see intro) -- ran end to end on this hardware without error.")
    print(f"{'='*70}")



CHECK FAILED -- 2/14 stage(s) did not complete cleanly:

  [TIMEOUT] Eval mixed-format checkpoint: MazeBench
  [TIMEOUT] Eval consistency-reward checkpoint: MazeBench


RuntimeError: 2 pipeline stage(s) failed or were skipped -- see the list above and the full cell output higher in this notebook for the actual error/traceback. Fix before trusting a real kaggle_train.ipynb run.

## Aggregate benchmark results into one comparison table

In [ ]:
import glob

rows = []
for path in sorted(glob.glob("./results_check_phase2/eval/*.json")):
    try:
        with open(path) as f:
            d = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Skipping unreadable results file {path}: {e}")
        continue
    label = path.split("/")[-1]
    # complete defaults True for older result files written before eval.py
    # saved incrementally -- those were only ever written once, at the end,
    # so a file existing at all meant it finished.
    row = {"file": label, "model": d.get("model"), "checkpoint": d.get("checkpoint") or "(none)",
           "benchmark": d.get("benchmark"), "n": d.get("n"), "n_completed": d.get("n_completed", d.get("n")),
           "complete": d.get("complete", True),
           "4bit": d.get("load_in_4bit")}  # AlphaMaze rows are always False (enforced in eval.py);
                                            # False Gemma 4 rows are the Phase 1b ablation
    if "mazebench" in d.get("benchmark", ""):
        row["score"] = d.get("accuracy")
        row["used_official_scoring"] = d.get("used_official_scoring")
    else:
        row["valid_rate"] = d.get("valid_rate")
        row["optimal_rate"] = d.get("optimal_rate")
    rows.append(row)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df.to_csv("./results_check_phase2/comparison_table.csv", index=False)
else:
    print("No eval result files found yet in ./results_check_phase2/eval/ -- nothing to aggregate.")
results_df


## Download results

Small, diagnostic-only -- this is the check run's tiny-sample output, not a substitute for `kaggle_train.ipynb`'s real results.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_check_phase2", "zip", "./results_check_phase2")
print("Saved: /kaggle/working/results_check_phase2.zip -- download it from the Kaggle output panel on the right.")
print("Includes stage_log.json (this run's full stage-by-stage status) and comparison_table.csv.")
